# Simple State Space Model


A simple State Space Model (SSM) hidden layer looks like:

\begin{align*}
h_{t+1} =&& A x_t + B x_t \\
y_t =&& C h_t + D x_t
\end{align*}    

<img src="ssm.png" alt="isolated" width="400"/>

Where x is the input signal, h the hidden state, and y the out put signal. A, B, C, B are parameters.

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

In [8]:
# ------------------------------------------------------------
# Simple SSM Layer using HiPPO initialization without skip connection D
# ------------------------------------------------------------
class SimpleSSM(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        
        # A starts close to identity but < 1
        A = torch.eye(state_dim) * 0.9
        A += 0.01 * torch.randn(state_dim, state_dim)
        self.A = nn.Parameter(A)
        self.B = nn.Parameter(torch.randn(state_dim, 1) * 0.01) # small B
        self.h0 = nn.Parameter(torch.zeros(state_dim)) # initial state
        self.C = nn.Linear(state_dim, 10)              # classifier
        self.norm = nn.LayerNorm(state_dim)

    def forward(self, x):
        # x shape: (batch, 784)
        batch = x.size(0)
        h = self.h0.unsqueeze(0).repeat(batch, 1)  # (batch, state_dim)

        for t in range(x.size(1)):
            u_t = x[:, t].unsqueeze(1)             # (batch, 1)
            h = h @ self.A.T + u_t @ self.B.T      # SSM update
            h = torch.tanh(h)                     # stabilizer
            h = torch.clamp(h, -5, 5)             # avoid blowup
        h = self.norm(h)      
        return self.C(h) # final hidden state → class logits

In [18]:
# ------------------------------------------------------------
# Load Sequential MNIST
# ------------------------------------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))    # flatten into (784,)
])

train = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train, batch_size=256, shuffle=True)
test_loader = DataLoader(test, batch_size=256)

print(len(train_loader), "batches")

235 batches


In [19]:
device = "mps" if torch.backends.mps.is_available() else  "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model = SimpleSSM(state_dim=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

mps


In [21]:
# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
for epoch in range(5):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        #torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    # test accuracy
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    print(f"Epoch {epoch+1}, accuracy = {correct/total:.4f}")


Epoch 1, accuracy = 0.2063
Epoch 2, accuracy = 0.4311
Epoch 3, accuracy = 0.4417
Epoch 4, accuracy = 0.4238
Epoch 5, accuracy = 0.5140


In [ ]:
################ Backup ##############

## HiPPO - High Order Polynomial Operator (Gu et al, 2020)

defines a continuous-time dynamical system whose state keeps an online projection of the past input signal onto the first N Legendre polynomials. This produces:
- A continuous-time state matrix A
- An input projection B
which together evolve the system:

$$h'(t) = Ah(t) + Bx(t)$$

S4 State Space Model takes these HiPPO matrices, discretizes them, and then optimizes them.

## HiPPO-LegS Matrix Formula

The most commonly referenced HiPPO matrix in S4 (S4-LegS) is:

\begin{align*}
         && &&-(2n+1)^{(1/2)} (2m+1)^{(1/2)} &&  n > m\\
A_{n,m}  &&=&&n && n = m\\
         && &&0 && n < m\\
B_{n}    &&=&&2n+1 &&
\end{align*} 



- It’s strictly lower triangular except the diagonal.
- It’s not symmetric.
- It grows with order → deeper polynomials get faster decay.

In [2]:
# ------------------------------------------------------------
# HiPPO-LegS matrix
# ------------------------------------------------------------
def hippo_legs(N):
    A = np.zeros((N, N))
    B = np.zeros(N)
    for n in range(N):
        for m in range(N):
            if n > m:
                A[n, m] = -np.sqrt((2*n+1)*(2*m+1))
            elif n == m:
                A[n, m] = n
            else:
                A[n, m] = 0
        B[n] = np.sqrt(2*n+1)
    return torch.tensor(A, dtype=torch.float32), torch.tensor(B, dtype=torch.float32)

In [3]:
hippo_legs(3)

(tensor([[ 0.0000,  0.0000,  0.0000],
         [-1.7321,  1.0000,  0.0000],
         [-2.2361, -3.8730,  2.0000]]),
 tensor([1.0000, 1.7321, 2.2361]))